# BTC1895H Final Exam — Student Template

This notebook is a **student-ready template** for the AI-allowed part of the mock/final exam.

## Files expected in the same folder
- `customers.csv`
- `activity_log.csv`
- `outcomes_train.csv`

## Reminder
- This is a **customer-level** prediction task.
- Explain your reasoning clearly.
- Do not rely only on model score.


## 1. Imports and setup

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)


## 2. Load the datasets

In [ ]:
customers = pd.read_csv("customers.csv", parse_dates=["signup_date"])
activity_log = pd.read_csv("activity_log.csv", parse_dates=["activity_date"])
outcomes = pd.read_csv("outcomes_train.csv", parse_dates=["target_snapshot_date"])

print("customers:", customers.shape)
print("activity_log:", activity_log.shape)
print("outcomes:", outcomes.shape)

customers.head()


## 3. Quick inspection

Write a few notes about what you observe:
- Which table is customer-level?
- Which table has repeated rows per customer?
- Which columns may need cleaning?


In [ ]:
customers.head(3), activity_log.head(3), outcomes.head(3)

**Your notes:**  
_Write your observations here._

## 4. Basic data checks

In [ ]:
for name, df in [("customers", customers), ("activity_log", activity_log), ("outcomes", outcomes)]:
    print(f"\n{name}")
    print(df.isna().sum().sort_values(ascending=False).head(10))


## 5. Aggregate the activity table to customer level

Create a customer-level summary from `activity_log`.

Suggested idea:
- counts
- averages
- sums
- latest or recent behavior if you want


In [ ]:
activity_agg = (
    activity_log
    .groupby("customer_id", as_index=False)
    .agg(
        n_events=("activity_id", "count"),
        avg_session_length=("session_length_min", "mean"),
        avg_pages_viewed=("pages_viewed", "mean"),
        total_cart_additions=("cart_additions", "sum"),
        total_promo_clicked=("promo_clicked", "sum"),
        total_coupon_shown=("coupon_shown", "sum"),
        total_coupon_redeemed=("coupon_redeemed", "sum"),
        total_support_chat=("support_chat_started", "sum"),
        total_refund_requested=("refund_requested", "sum"),
        total_shipping_delay=("shipping_delay_flag", "sum"),
    )
)

activity_agg.head()


## 6. Merge the tables

Create a modeling dataset at the **customer level**.


In [ ]:
model_df = (
    customers
    .merge(activity_agg, on="customer_id", how="left")
    .merge(outcomes, on="customer_id", how="left")
)

print(model_df.shape)
model_df.head()


## 7. Think before modeling

Before you train a model, answer these:

1. Are there any columns that look suspicious?
2. Are there any columns that may contain future information?
3. Is a random split always appropriate here?

Write your thoughts below.


**Your reasoning:**  
_Write your answer here._

## 8. Choose features

Edit the list below based on your reasoning.


In [ ]:
target = "purchased_next_30d"

drop_cols = [
    "customer_id",
    target,
    # Add any columns you decide should not be used for training
]

X = model_df.drop(columns=drop_cols, errors="ignore")
y = model_df[target]

X.head()


## 9. Train / validation split

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(X_train.shape, X_valid.shape)


## 10. Preprocessing + model

In [ ]:
numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_cols),
    ("cat", categorical_pipe, categorical_cols)
])

model = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)
pred = model.predict(X_valid)

acc = accuracy_score(y_valid, pred)
f1 = f1_score(y_valid, pred)

print("Accuracy:", round(acc, 4))
print("F1 score:", round(f1, 4))
print()
print(classification_report(y_valid, pred))


## 11. Optional comparison model

In [ ]:
rf_model = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_valid)

rf_acc = accuracy_score(y_valid, rf_pred)
rf_f1 = f1_score(y_valid, rf_pred)

print("RandomForest Accuracy:", round(rf_acc, 4))
print("RandomForest F1 score:", round(rf_f1, 4))


## 12. Interpretation

Answer these questions in complete sentences:

1. What model did you choose and why?
2. What score did you get?
3. Which variables seem important or suspicious?
4. What data issues did you notice?
5. How might your validation strategy affect the result?


**Your interpretation:**  
_Write your answer here._

## 13. AI reflection

If you used AI, answer:

1. What did AI help you do?
2. What did AI miss or oversimplify?
3. What did you correct manually?


**Your AI reflection:**  
_Write your answer here._